# PEBBLE H1/H2 on Colab (A100)

**Study:** when PEBBLE true return drops under paper ablations, is it reward-model ranking (**H1**) or SAC amplification (**H2**)?  
**Pre-registration:** `experiments/pebble_reward_vs_rl/CLAIM.md`  
**Not B-Pref** — Oracle teacher only (Lee et al., ICML 2021).

## Setup (do this first)
1. **Runtime → Change runtime type → GPU → A100** (Colab Pro / Pro+; not guaranteed on free).
2. **Runtime → Disconnect and delete runtime**, then reconnect if you previously half-installed packages.
3. Run cells top-to-bottom. Keep **Drive mounted** so long suites survive disconnects.

## Python / gym note
Colab may use Python 3.12+. `gym==0.26.2` (validated stack) often fails to build there.  
**We do not fall back to a newer gym** — that would silently change env APIs.  
Cell 4 bootstraps a **Python 3.11 venv** via `uv` when needed and pins `gym==0.26.2`.

## Status vocabulary
| Stage | This notebook |
|-------|----------------|
| Smoke | wiring only — not evidence |
| Diagnostic (5×4×100k) | tentative patterns only |
| Paper-scale (10×4×≥500k) | claim gate (CLAIM.md §6) |

In [ ]:
# Cell 1 — GPU check (prefer A100)
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → GPU → A100 (or any NVIDIA GPU)."
)
name = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} ({mem_gb:.1f} GB)")

is_a100 = "A100" in name.upper()
if not is_a100:
    print(
        "WARNING: This is not an A100. Suites still run on T4/L4/V100, but slower.\n"
        "Reconnect / change runtime until you get A100 if that is required."
    )
else:
    print("A100 detected — good for diagnostic + paper-scale.")

In [ ]:
# Cell 2 — Drive persistence (recommended for multi-hour suites)
from pathlib import Path

USE_DRIVE = True  # set False only for short smoke tests
REPO_URL = "https://github.com/thatrandomasiandev/BPref.git"
REPO_BRANCH = "main"

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    ROOT = Path("/content/drive/MyDrive/LiraLab/BPref")
else:
    ROOT = Path("/content/BPref")

ROOT.parent.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)

In [ ]:
# Cell 3 — Clone / update the experiment fork (includes pebble_reward_vs_rl)
import os
import subprocess
from pathlib import Path

def sh(cmd, cwd=None):
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True, cwd=cwd)

if (ROOT / ".git").exists():
    sh(
        f"git fetch origin {REPO_BRANCH} && git checkout {REPO_BRANCH} && git pull --ff-only origin {REPO_BRANCH}",
        cwd=str(ROOT),
    )
elif ROOT.exists() and any(ROOT.iterdir()):
    raise RuntimeError(f"{ROOT} exists but is not a git repo. Move/rename it, then re-run.")
else:
    sh(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {ROOT}")

assert (ROOT / "experiments/pebble_reward_vs_rl/CLAIM.md").exists(), (
    "CLAIM.md missing — wrong repo/branch. Need thatrandomasiandev/BPref with pebble_reward_vs_rl."
)
os.chdir(ROOT)
print("cwd:", Path.cwd())
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
# Cell 4 — Dependencies (pin gym==0.26.2; use Python 3.11 venv on Colab 3.12+)
#
# Do NOT fall back to "latest gym" — that silently changes env APIs and breaks
# the validated PEBBLE instrument. If system Python is >= 3.12, bootstrap 3.11.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = f"{ROOT}:{ROOT / 'custom_dmc2gym'}"

PKGS = [
    "hydra-core",
    "omegaconf",
    "gym==0.26.2",  # pinned — update CLAIM.md §8 if this ever changes
    "dm_control",
    "mujoco",
    "scikit-image",
    "tensorboard",
    "tqdm",
    "matplotlib",
]


def _run(cmd, **kwargs):
    print(">>", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd, **kwargs)


def _ensure_uv() -> str:
    uv = shutil.which("uv")
    if uv:
        return uv
    _run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True)
    for c in [
        Path.home() / ".local" / "bin" / "uv",
        Path.home() / ".cargo" / "bin" / "uv",
        Path("/root/.local/bin/uv"),
    ]:
        if c.exists():
            os.environ["PATH"] = f"{c.parent}:{os.environ.get('PATH', '')}"
            return str(c)
    uv = shutil.which("uv")
    assert uv, "uv install failed"
    return uv


print(f"system python: {sys.version}")
need_venv = sys.version_info >= (3, 12)
VENV_DIR = Path("/content/pebble311")

if need_venv:
    print("System Python >= 3.12 → Python 3.11 venv for gym==0.26.2")
    uv = _ensure_uv()
    _run([uv, "python", "install", "3.11"])
    if not (VENV_DIR / "bin" / "python").exists():
        _run([uv, "venv", str(VENV_DIR), "--python", "3.11"])
    PEBBLE_PYTHON = str(VENV_DIR / "bin" / "python")
    # System Colab torch is not visible inside the venv — install CUDA torch.
    _run([
        uv, "pip", "install", "--python", PEBBLE_PYTHON, "torch",
        "--index-url", "https://download.pytorch.org/whl/cu124",
    ])
    _run([uv, "pip", "install", "--python", PEBBLE_PYTHON, *PKGS])
    _run(
        [uv, "pip", "install", "--python", PEBBLE_PYTHON, "-e", "custom_dmc2gym"],
        cwd=str(ROOT),
    )
else:
    print("System Python < 3.12 → install into the Colab runtime")
    PEBBLE_PYTHON = sys.executable
    try:
        _run([PEBBLE_PYTHON, "-m", "pip", "install", "-q", "gym==0.26.2"])
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            "gym==0.26.2 failed. Do NOT silently upgrade gym. "
            "Disconnect runtime and re-run, or force need_venv=True above."
        ) from e
    other = [p for p in PKGS if not p.startswith("gym")]
    _run([PEBBLE_PYTHON, "-m", "pip", "install", "-q", *other])
    _run(
        [PEBBLE_PYTHON, "-m", "pip", "install", "-q", "-e", "custom_dmc2gym"],
        cwd=str(ROOT),
    )

os.environ["PEBBLE_PYTHON"] = PEBBLE_PYTHON
print("PEBBLE_PYTHON =", PEBBLE_PYTHON)

ver = subprocess.check_output(
    [
        PEBBLE_PYTHON, "-c",
        "import gym, torch, sys; print(gym.__version__, torch.__version__, torch.cuda.is_available(), sys.version.split()[0])",
    ],
    text=True,
).strip()
print("verify:", ver)
gym_ver = ver.split()[0]
assert gym_ver.startswith("0.26"), f"Expected gym 0.26.x, got {gym_ver}"
assert "True" in ver.split(), f"CUDA not visible in PEBBLE_PYTHON: {ver}"
print("MUJOCO_GL", os.environ["MUJOCO_GL"])
print("PYTHONPATH", os.environ["PYTHONPATH"])

In [ ]:
# Cell 5 — Smoke (wiring only — NOT evidence)
import os
import subprocess
from pathlib import Path

os.chdir(ROOT)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONPATH"] = f"{ROOT}:{ROOT / 'custom_dmc2gym'}"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["CONDITION"] = "full"
os.environ["DEVICE"] = "cuda"
os.environ["SEED"] = "0"
os.environ["STEPS"] = "6000"
os.environ["ENV"] = "walker_walk"

py = os.environ.get("PEBBLE_PYTHON", "python")
smoke_dir = "experiments/pebble_reward_vs_rl/exp/walker_walk/full/seed0_colab_smoke"
cmd = [
    py,
    "experiments/pebble_reward_vs_rl/train_pebble_diagnostics.py",
    "device=cuda",
    "seed=0",
    "num_train_steps=6000",
    "env=walker_walk",
    "pebble_condition=full",
    "do_relabel=true",
    "num_seed_steps=1000",
    "num_unsup_steps=2000",
    "num_interact=2000",
    "max_feedback=80",
    "reward_batch=20",
    "reward_update=5",
    "eval_frequency=4000",
    "num_eval_episodes=1",
    "diag_holdout_pairs=64",
    "diag_onpolicy_pairs=32",
    "diag_probe_gradient_steps=20",
    "diag_probe_steps=[6000]",
    "agent.batch_size=256",
    f"hydra.run.dir={smoke_dir}",
]
print("SMOKE (wiring only):", " ".join(cmd))
subprocess.check_call(cmd, cwd=str(ROOT))
csv = Path(smoke_dir) / "diagnostics.csv"
assert csv.exists(), csv
print("Smoke OK —", csv)
print("This is NOT a scientific finding.")

In [ ]:
# Cell 6 — Diagnostic suite (CLAIM.md §6: tentative patterns only)
import os
import subprocess

os.chdir(ROOT)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONPATH"] = f"{ROOT}:{ROOT / 'custom_dmc2gym'}"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["DEVICE"] = "cuda"
os.environ["PARALLEL"] = "1"
os.environ["STEPS"] = "100000"
os.environ["SEEDS"] = "1 2 3 4 5"
os.environ["ENV"] = "walker_walk"

py = os.environ.get("PEBBLE_PYTHON", "python")
print("Starting DIAGNOSTIC suite with", py)
rc = subprocess.call(
    [py, "experiments/pebble_reward_vs_rl/run_diagnostic_suite.py"],
    cwd=str(ROOT),
)
print("diagnostic suite exit:", rc)
if rc != 0:
    raise SystemExit(f"Diagnostic suite failed with rc={rc}. Check exp/_logs/.")

In [ ]:
# Cell 7 — Aggregate diagnostic results
import os
import subprocess
from pathlib import Path

os.chdir(ROOT)
py = os.environ.get("PEBBLE_PYTHON", "python")
subprocess.check_call(
    [
        py,
        "experiments/pebble_reward_vs_rl/analyze_results.py",
        "--root",
        "experiments/pebble_reward_vs_rl/exp",
    ],
    cwd=str(ROOT),
)
summary = Path("experiments/pebble_reward_vs_rl/exp/_analysis/summary_by_condition.csv")
print(summary.read_text() if summary.exists() else "missing summary")
print("\nSTATUS: diagnostic aggregation only — not a justified H1/H2 conclusion.")

In [ ]:
# Cell 8 — Paper-scale suite (CLAIM.md claim gate)
import os
import subprocess

RUN_PAPER_SCALE = True  # set False to stop after diagnostic

if not RUN_PAPER_SCALE:
    print("Skipping paper-scale (RUN_PAPER_SCALE=False).")
else:
    os.chdir(ROOT)
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYTHONPATH"] = f"{ROOT}:{ROOT / 'custom_dmc2gym'}"
    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ["DEVICE"] = "cuda"
    os.environ["PARALLEL"] = "1"
    os.environ["STEPS"] = "500000"
    os.environ["SEEDS"] = "1 2 3 4 5 6 7 8 9 10"
    os.environ["ENV"] = "walker_walk"
    py = os.environ.get("PEBBLE_PYTHON", "python")
    print("Starting PAPER-SCALE suite with", py)
    rc = subprocess.call(
        [py, "experiments/pebble_reward_vs_rl/run_paper_scale_suite.py"],
        cwd=str(ROOT),
    )
    print("paper-scale exit:", rc)
    if rc != 0:
        raise SystemExit(f"Paper-scale failed rc={rc}. Check exp/_logs/.")
    subprocess.check_call(
        [
            py,
            "experiments/pebble_reward_vs_rl/analyze_results.py",
            "--root",
            "experiments/pebble_reward_vs_rl/exp",
        ],
        cwd=str(ROOT),
    )

## After runs

- Results live under `experiments/pebble_reward_vs_rl/exp/` (on Drive if `USE_DRIVE=True`).
- Fill `RESULTS.md` from `_analysis/summary_by_condition.csv` using CLAIM.md §7.
- **Do not** claim H1/H2 from smoke or incomplete seed sets.

If Colab disconnects: reconnect the **same** GPU type, re-run cells 1–4, then Cell 6 or 8 — suites skip completed `seed*` folders.

If Cell 4 fails on gym: **do not** install unpinned gym. Disconnect runtime and re-run Cell 4 so the 3.11 venv path is used.